# v01 — Zero-shot baseline

End-to-end Colab runner for `v01_zeroshot_baseline`. All knobs (model, dtype, batch size, max tokens, ...) live in [`app/physics_solution/config.py`](../../config.py) — edit there, not here.

Runs **directly off the Drive copy** (Colab Oct-2025+ runtime is fast enough that copying to `/content/` is no longer worth the disk churn). The repo path `REPO_ROOT` below points to your Drive folder.

## 1. Mount Drive + chdir


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
REPO_ROOT = '/content/drive/MyDrive/NCKH/Exact_2026_Laplace-s_Red_Devils'  # change if needed
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print('cwd:', os.getcwd())

In [ ]:
!pip -q install -r app/physics_solution/requirements.txt

### Install Qwen3.5 fast-attention kernels (`flash-linear-attention` + `causal-conv1d`)

On Colab Oct-2025+ runtimes both wheels install cleanly. If `causal_conv1d: False` shows up, the model still runs via torch fallback at ~3× slower — accuracy is unaffected.

In [ ]:
from app.physics_solution.shared.colab.setup import install_fast_kernels
install_fast_kernels()

In [ ]:
# setup_colab handles .env + LangSmith.
from app.physics_solution.shared.colab.setup import setup_colab
setup_colab(repo_root=REPO_ROOT)

In [ ]:
!nvidia-smi -L

## 2. Build test sample

By default we run on **all 973 pure-numeric questions** for proper eval. Re-roll with a different seed by deleting `sample_test.csv` first.

In [ ]:
import os
SAMPLE = 'app/physics_solution/data/sample_test.csv'
if not os.path.exists(SAMPLE):
    !python -m app.physics_solution.data.prepare_sample --n 973 --seed 42 --out {SAMPLE}
else:
    print('sample exists:', SAMPLE)

## 3. Inference

All knobs (batch size, dtype, max_new_tokens, ...) come from `config.py`. Add e.g. `--limit 50 --batch-size 8` to override per-run for smoke tests.

In [ ]:
!python -m app.physics_solution.inference --version v01_zeroshot_baseline

## 4. Push to HF (with full metadata)


In [ ]:
!python -m app.physics_solution.upload_model \
    --version-num 1 --strategy zeroshot \
    --results app/physics_solution/results/v01_zeroshot.json

## 5. Inspect results


In [ ]:
import json, pandas as pd
data = json.loads(open('app/physics_solution/results/v01_zeroshot.json', encoding='utf-8').read())
print('summary:', json.dumps(data['summary'], indent=2, ensure_ascii=False))
df = pd.DataFrame(data['rows'])
wrong = df[~df['is_correct']].copy()
wrong['reached_final'] = wrong['completion'].str.contains('FINAL ANSWER', case=False, regex=False)
print(f"\nWrong: {len(wrong)} / {len(df)}")
print(f"  ... never reached FINAL ANSWER: {(~wrong['reached_final']).sum()}")
wrong[['id', 'gold_answer', 'pred_numeric', 'pred_unit', 'gold_unit', 'reached_final']].head(20)

## 6. Deep error analysis

Open [`app/physics_solution/eda/error_analysis.ipynb`](../../eda/error_analysis.ipynb) — it categorises every wrong row into fail modes (format / order-of-magnitude / unit / domain-specific / suspected-gold-error) and writes a markdown report.